In [1]:
!pip install -U requests pyarrow pandas datasets fastparquet pydantic tqdm


In [1]:
import itertools
import json
import csv
import asyncio
import re
import ollama
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from typing import List, Optional
from pydantic import BaseModel, Field
from tqdm.auto import tqdm


# Configurção

In [2]:
from pydantic import BaseModel, Field, field_validator
from typing import List

MODELO = "llama3"
CSV_FILE = "dataset_clean.csv"
OUTPUT_DIR = "batches_temp"
FINAL_DATASET = "dataset_finetuning.parquet"

BATCH_SIZE = 50
CONCURRENCY = 8
LIMIT_ITEM = 3

Path(OUTPUT_DIR).mkdir(exist_ok=True)

# Modelos de validação rigorosa
class Issue(BaseModel):
    title: str = Field(..., max_length=100)
    body: str = Field(...)
    labels: List[str] = Field(default_factory=lambda: ["technical-debt"])

    @field_validator('body')
    @classmethod
    def clean_newlines_and_validate(cls, v: str) -> str:
        # Converte \n (literal) para (enter) caso a IA tenha escapado
        v = v.replace('\\n', '\n')
        v = v.replace('\n', '\n') # Garante consistência
        if "###" not in v:
            # Se não tiver markdown, tenta estruturar minimamente
            v = f"### Objetivo\n{v}"
        return v

class IssueList(BaseModel):
    issues: List[Issue]

PROMPT_TEMPLATE = """
Você é um engenheiro sênior. Transforme o escopo em issues do GitHub.\n
### Formato Obrigatório (JSON):\n
{{\n
  \"issues\": [\n
    {{\n
      \"title\": \"Título Conciso\",\n
      \"body\": \"### Objetivo\\n...\\n\\n### Descrição Técnica\\n...\\n\\n### Tasks\\n- [ ] Task 1\",\n
      \"labels\": [\"backend\", \"api\"]\n
    }}\n
  ]\n
}}\n
### Instruções:\n
- Use APENAS JSON.\n
- Use Markdown (###) no body.\n
- As issues devem ser estruturadas de forma que o desenvolvedor possa entender o escopo e as tarefas que devem ser realizadas para concluir o escopo.\n
- As descrições devem ser detalhadas e clara para o desenvolvedor.\n
- Nos Headers do body(marcados por ###) e nos textos gerados use a lingua que foi solicitada no escopo(ou se não definida, a lingua utilizada no escopo).\n
- Máximo 100 caracteres no title.\n
- foco em arquitetura, backend, frontend, banco de dados, APIs, infraestrutura, CI/CD e testes\n
### Escopo:\n
{escopo}
"""

sem = asyncio.Semaphore(CONCURRENCY)


In [3]:
def formatar_para_treino(input_text, output):

    instruction = "Read the software project description and produce structured GitHub issues that represent the core engineering tasks needed to build the system, including architecture, backend, frontend, APIs, infrastructure, testing, and deployment."

    response = json.dumps(output, ensure_ascii=False)

    formatted = f"""### Instruction:
{instruction}

### Input:
{input_text}

### Response:
{response}"""

    return formatted


# Gerar Issues

In [101]:
# from google import genai
#
# client = genai.Client(api_key="TOKEN GEMINI")

In [4]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    async with sem:
        prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
        for tentativa in range(3):
            try:
                response = ollama.generate(model=MODELO, prompt=prompt, format="json")
                raw_json = extrair_json(response.get("response", ""))
                if raw_json:
                    # Validação e Limpeza Pydantic
                    if isinstance(raw_json, list):
                        data = IssueList(issues=raw_json)
                    else:
                        data = IssueList(**raw_json)
                    
                    # O .dict() do pydantic v2/v1 já deve ter processado o validator
                    return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
            except Exception as e:
                if tentativa == 2: print(f"Erro na IA: {e}")
        return []


In [5]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    async with sem:
        prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
        for tentativa in range(3):
            try:
                response = ollama.generate(model=MODELO, prompt=prompt, format="json")
                raw_json = extrair_json(response.get("response", ""))
                if raw_json:
                    # Validação e Limpeza Pydantic
                    if isinstance(raw_json, list):
                        data = IssueList(issues=raw_json)
                    else:
                        data = IssueList(**raw_json)
                    
                    # O .dict() do pydantic v2/v1 já deve ter processado o validator
                    return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
            except Exception as e:
                if tentativa == 2: print(f"Erro na IA: {e}")
        return []


In [6]:
TESTE = """
Criar uma plataforma inteligente que utilize IA para processar documentação (como PDFs) e automatizar a conversão de requisitos técnicos e de negócio em issues estruturadas no GitHub. O sistema deve transformar documentos brutos em tarefas organizadas, com títulos claros, descrições técnicas detalhadas, critérios de aceitação e categorização automática (Feature, Bug, Improvement, Technical Task).

A plataforma deve:

* Automatizar a ingestão de dados, convertendo PDFs em issues completas.
* Padronizar a organização do backlog com aplicação automática de labels e segmentações.
* Gerar estimativas de tempo total de produção e sugerir divisão em sprints com base na densidade do backlog.
* Extrair e documentar requisitos funcionais, não-funcionais, regras de negócio e restrições técnicas.
* Identificar inconsistências entre documentos e sugerir melhorias arquiteturais por meio de relatórios executivos sob demanda.
* Reduzir o gap de comunicação entre negócio e desenvolvimento, garantindo que as tarefas reflitam fielmente os objetivos estratégicos da empresa.
"""

In [7]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    async with sem:
        prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
        for tentativa in range(3):
            try:
                response = ollama.generate(model=MODELO, prompt=prompt, format="json")
                raw_json = extrair_json(response.get("response", ""))
                if raw_json:
                    # Validação e Limpeza Pydantic
                    if isinstance(raw_json, list):
                        data = IssueList(issues=raw_json)
                    else:
                        data = IssueList(**raw_json)
                    
                    # O .dict() do pydantic v2/v1 já deve ter processado o validator
                    return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
            except Exception as e:
                if tentativa == 2: print(f"Erro na IA: {e}")
        return []


In [13]:
async def processar_item(texto):

    issues = await gerar_issue_async(texto)

    if not issues:
        return None

    texto_treino = formatar_para_treino(texto, issues)

    print(texto_treino)

    return {"text": texto_treino}

In [8]:
def extrair_json(texto):
    try:
        return json.loads(texto)
    except json.JSONDecodeError:
        match = re.search(r'(\[.*?\]|\{.*?\})', texto, re.DOTALL)
        if match: 
            try: return json.loads(match.group(1))
            except: pass
    return None

async def gerar_issue_async(texto):
    async with sem:
        prompt = PROMPT_TEMPLATE.format(escopo=texto[:2500])
        for tentativa in range(3):
            try:
                response = ollama.generate(model=MODELO, prompt=prompt, format="json")
                raw_json = extrair_json(response.get("response", ""))
                if raw_json:
                    # Validação e Limpeza Pydantic
                    if isinstance(raw_json, list):
                        data = IssueList(issues=raw_json)
                    else:
                        data = IssueList(**raw_json)
                    
                    # O .dict() do pydantic v2/v1 já deve ter processado o validator
                    return [i.model_dump() if hasattr(i, 'model_dump') else i.dict() for i in data.issues]
            except Exception as e:
                if tentativa == 2: print(f"Erro na IA: {e}")
        return []


In [9]:
def salvar_batch(batch, index):

    file = f"{OUTPUT_DIR}/dataset_batch_{index}.json"

    with open(file, "w", encoding="utf-8") as f:
        json.dump(batch, f, ensure_ascii=False)

    print("Batch salvo:", file)


In [10]:
def consolidar_parquet():

    print("\nConsolidando dataset final...")

    rows = []

    for file in Path(OUTPUT_DIR).glob("*.json"):

        with open(file, "r", encoding="utf-8") as f:
            data = json.load(f)

            for item in data:

                if not isinstance(item, dict):
                    continue

                text = item.get("text")

                if not text:
                    continue

                rows.append({"text": str(text)})

    if not rows:
        print("Nenhum exemplo válido encontrado.")
        return

    df = pd.DataFrame(rows)

    print("Total exemplos válidos:", len(df))

    # força tipo string
    df = df.astype({"text": "string"})

    # remove possíveis valores inválidos
    df = df.dropna()

    df.to_parquet(
        FINAL_DATASET,
        engine="pyarrow",
        compression="snappy",
        index=False
    )

    print("Dataset final salvo:", FINAL_DATASET)

In [109]:
# consolidar_parquet()


Consolidando dataset final...
Nenhum exemplo válido encontrado.


In [18]:
async def main():
    textos = []
    print("Carregando CSV...")
    with open(CSV_FILE, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
        #for row in itertools.islice(reader, LIMIT_ITEM):
            if row.get("descricao"):
                textos.append(row["descricao"])

    print(f"Processando {len(textos)} descrições...")
    batch, batch_index, tasks = [], 0, []
    pbar = tqdm(total=len(textos), desc="Gerando Issues")

    for i, texto in enumerate(textos):
        tasks.append(processar_item(texto))
        if len(tasks) >= CONCURRENCY or i == len(textos) - 1:
            resultados = await asyncio.gather(*tasks)
            for r in resultados:
                if r: batch.append(r)
                if len(batch) >= BATCH_SIZE:
                    salvar_batch(batch, batch_index)
                    batch, batch_index = [], batch_index + 1
            pbar.update(len(tasks))
            tasks = []
    if batch: salvar_batch(batch, batch_index)
    pbar.close()
    consolidar_parquet()


In [19]:
await main()

Carregando CSV...
Processando 665 descrições...


Gerando Issues:   0%|          | 0/665 [00:00<?, ?it/s]

### Instruction:
Read the software project description and produce structured GitHub issues that represent the core engineering tasks needed to build the system, including architecture, backend, frontend, APIs, infrastructure, testing, and deployment.

### Input:
HealthBridge is a microservices-based telemedicine and remote patient monitoring platform designed to provide healthcare professionals and patients with a secure, user-friendly solution for virtual consultations, remote monitoring, and care coordination. The platform comprises several microservices, including video conferencing, electronic health record integration, remote device monitoring, and secure messaging. By adopting a microservices architecture, HealthBridge offers a highly scalable and customizable solution that helps improve patient outcomes, expand healthcare access, and reduce healthcare costs.

### Response:
[{"title": "HealthBridge Platform Architecture", "body": "### Objective\nDesign the overall architecture f

In [20]:
import csv, json
def exportar_para_csv(parquet_path, output_csv):
    print(f"Exportando para {output_csv} com quebras de linha reais...")
    table = pq.read_table(parquet_path)
    df = table.to_pandas()
    
    with open(output_csv, "w", newline="", encoding="utf-8") as f:
        # lineterminator='\n' garante que o enter seja o padrão Unix/Universal
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        writer.writerow(["Title", "Description"])

        for text in df["text"].dropna():
            if "### Response:" not in text: continue
            try:
                json_str = text.split("### Response:")[1].strip()
                data = json.loads(json_str)
                issues = data.get("issues", data) if isinstance(data, dict) else data
                
                for issue in issues:
                    if not isinstance(issue, dict): continue
                    
                    title = issue.get("title", "")
                    body = issue.get("body", "")
                    
                    # Força a limpeza final de escapes de string literais
                    # O decode('unicode_escape') pode ser perigoso, melhor substituição direta
                    body = body.replace("\\n", "\n")
                    
                    writer.writerow([title, body])
            except:
                pass

exportar_para_csv(FINAL_DATASET, "gitlab_issues.csv")


Exportando para gitlab_issues.csv com quebras de linha reais...
